# Sentiment Analysis – Datenanalyse

Dieses Notebook lädt einen Sentiment-Datensatz (Kaggle oder Dummy-Fallback), analysiert ihn mit **pandas**, visualisiert die Verteilung mit **matplotlib** und testet eine erste **HuggingFace** Sentiment-Pipeline.

## 1. Imports & Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_colwidth', 80)
plt.style.use('default')

## 2. Datensatz laden

Versuche zuerst, den Kaggle-Datensatz zu laden. Falls nicht verfügbar, wird der mitgelieferte Dummy-Datensatz verwendet.

In [ ]:
DUMMY_PATH = Path('../data/sentiment_data.csv')
KAGGLE_PATH = Path('../data/kaggle_sentiment.csv')  # optional

def load_dataset():
    if KAGGLE_PATH.exists():
        print(f'Lade Kaggle-Datensatz aus {KAGGLE_PATH}')
        return pd.read_csv(KAGGLE_PATH)
    print(f'Kaggle-Daten nicht gefunden, lade Dummy-Daten aus {DUMMY_PATH}')
    return pd.read_csv(DUMMY_PATH)

df = load_dataset()
df.head()

## 3. Explorative Datenanalyse

In [ ]:
print(f'Anzahl Zeilen: {len(df)}')
print(f'Spalten: {list(df.columns)}')
print('\nLabel-Verteilung:')
print(df['label'].value_counts())
print('\nFehlende Werte:')
print(df.isna().sum())

In [ ]:
df['text_length'] = df['text'].str.len()
df.groupby('label')['text_length'].agg(['mean', 'min', 'max']).round(1)

## 4. Visualisierung – Sentiment Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['label'].value_counts()
colors = ['#4CAF50', '#F44336', '#FFC107']

axes[0].bar(counts.index, counts.values, color=colors[:len(counts)])
axes[0].set_title('Sentiment Distribution (Bar)')
axes[0].set_ylabel('Anzahl')

axes[1].pie(counts.values, labels=counts.index, autopct='%1.1f%%', colors=colors[:len(counts)], startangle=90)
axes[1].set_title('Sentiment Distribution (Pie)')

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for label in df['label'].unique():
    subset = df[df['label'] == label]
    ax.hist(subset['text_length'], alpha=0.6, label=label, bins=15)
ax.set_xlabel('Textlänge (Zeichen)')
ax.set_ylabel('Häufigkeit')
ax.set_title('Textlänge je Sentiment')
ax.legend()
plt.show()

## 5. HuggingFace Sentiment-Pipeline testen

Erster Test mit dem vortrainierten DistilBERT-Modell (SST-2).

In [ ]:
from transformers import pipeline

classifier = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english'
)
print('Pipeline geladen!')

In [ ]:
samples = df['text'].sample(5, random_state=42).tolist()
predictions = classifier(samples)

for text, pred in zip(samples, predictions):
    print(f"[{pred['label']:8s} {pred['score']:.2%}]  {text}")

In [ ]:
df_eval = df[df['label'].isin(['positive', 'negative'])].copy()
df_eval['prediction'] = [p['label'].lower() for p in classifier(df_eval['text'].tolist())]
accuracy = (df_eval['label'] == df_eval['prediction']).mean()
print(f'Accuracy auf Dummy-Daten (positive/negative): {accuracy:.2%}')

## 6. Fazit

- Der Datensatz ist über pandas leicht zu explorieren.
- DistilBERT erreicht auf dem Dummy-Set eine sehr hohe Accuracy für `positive`/`negative`.
- Nächster Schritt: eigenen Klassifikator fine-tunen oder Modell mit Kaggle-Daten evaluieren.